# Juiz de Bocha — teste visual do modelo treinado

Carrega os modelos treinados — **A100** (`00-Juiz de Bocha/treinoA100`) e **G4** (`00-Juiz de Bocha/treino`) — e abre uma tela onde você **arrasta uma foto da jogada** (ou tira com a câmera) e vê:
- **Análise:** cada bola numerada pela distância ao balim — branco = balim, verde = vencedora, amarelo = demais;
- **Animação do app:** o mesmo GIF que a API do Cloud Run devolve (`cloud-run-function/main.py`);
- **Comparar modelos:** a mesma foto julgada pelos dois modelos lado a lado;
- **Tabela** com score, área e distância de cada bola.

No fim tem uma **galeria** que roda os modelos em várias fotos de uma pasta do Drive de uma vez (uma coluna por modelo). Se um dos modelos ainda não existir no Drive, ele é ignorado e o notebook segue com o outro.

**Antes de rodar:** *Ambiente de execução → Alterar tipo → GPU* (T4 basta para teste).

## 1. Ambiente

In [ ]:
!nvidia-smi -L || echo "Sem GPU — vai funcionar, mas cada foto leva alguns segundos"
try:
    import detectron2
except ImportError:
    !pip install -q exifread
    !pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q gradio

import os, sys
from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = "/content/juiz-de-bocha-algoritmo"
if not os.path.isdir(REPO_DIR):
    !git clone -q https://github.com/JonathasmpDEV/juiz-de-bocha-algoritmo.git {REPO_DIR}
sys.path.insert(0, REPO_DIR)

## 2. Carregar os modelos

In [ ]:
import torch
from recognizer import Recognizer

MODELO_A100 = "/content/drive/MyDrive/00-Juiz de Bocha/treinoA100/modelo_final"  # @param {type:"string"}
MODELO_G4 = "/content/drive/MyDrive/00-Juiz de Bocha/treino/modelo_final"  # @param {type:"string"}

# carrega com limiar baixo; o limiar de verdade é aplicado depois (slider na tela)
modelos = {}
for nome, pasta in (("A100", MODELO_A100), ("G4", MODELO_G4)):
    if not os.path.exists(f"{pasta}/model_final.pth"):
        print(f"✘ {nome}: não encontrado em {pasta} (treino ainda não terminou?)")
        continue
    modelos[nome] = Recognizer(
        config_file=f"{pasta}/config.yaml",
        weights_file=f"{pasta}/model_final.pth",
        confidence_threshold=0.3,
        gpu_id=0 if torch.cuda.is_available() else None,
    )
    print(f"✔ {nome}: {pasta}")
assert modelos, "Nenhum modelo encontrado — confira os caminhos acima."
print("GPU:", torch.cuda.is_available())

## 3. Lógica do juiz e desenhos
`processar_bolas` é a mesma regra de `__process_balls`; `imagens_do_app` e `gif_do_app` reproduzem `_process_image` e `_to_gif_bytes` de `cloud-run-function/main.py` (bolas pintadas com a cor média, círculo verde piscando na vencedora, cruz no balim).

In [ ]:
import io, math, time
import numpy as np
import cv2
import skimage.draw
from PIL import Image

def processar_bolas(img, instancias):
    instancias = [i for i in instancias if i["class"] == "sports ball"]
    bolas, balim, vencedora = [], None, None
    if len(instancias) >= 2:
        margem = 0.10
        x_min, x_max = img.shape[1] * margem, img.shape[1] * (1 - margem)
        y_min, y_max = img.shape[0] * margem, img.shape[0] * (1 - margem)
        no_meio, nas_bordas = [], []
        for inst in instancias:
            c = inst["center"]
            (no_meio if x_min <= c["x"] <= x_max and y_min <= c["y"] <= y_max else nas_bordas).append(inst)
        if no_meio:
            no_meio.sort(key=lambda i: i["area"])
            while True:
                balim = no_meio.pop(0)
                if not no_meio:
                    break
                if math.fabs(no_meio[-1]["area"] / balim["area"] - no_meio[0]["area"] / balim["area"]) <= 1.5:
                    break
            nas_bordas = [b for b in nas_bordas if b["area"] >= balim["area"] / 2]
            bolas = no_meio + nas_bordas
            for b in bolas:
                b["distance"] = math.hypot(b["center"]["x"] - balim["center"]["x"], b["center"]["y"] - balim["center"]["y"])
            bolas.sort(key=lambda b: b["distance"])
            vencedora = bolas[0] if bolas else None
    return bolas, vencedora, balim

def imagem_analise(img, todas, bolas, vencedora, balim):
    out = img.copy()
    esp = max(3, int(max(img.shape[:2]) * 0.004))
    fonte = max(0.8, max(img.shape[:2]) / 1200)
    def elipse(inst, cor, extra=0, e=esp):
        b = inst["box"]
        cv2.ellipse(out, (int(inst["center"]["x"]), int(inst["center"]["y"])),
                    (int((b["x2"] - b["x1"]) / 2) + extra, int((b["y2"] - b["y1"]) / 2) + extra), 0, 0, 360, cor, e)
    usadas = {id(b) for b in bolas} | ({id(balim)} if balim else set())
    for inst in todas:  # detecções descartadas pela regra (sujeira, bordas)
        if id(inst) not in usadas:
            elipse(inst, (150, 150, 150), e=max(1, esp // 2))
    for n, b in enumerate(bolas, 1):
        cor = (50, 230, 50) if b is vencedora else (255, 200, 0)
        elipse(b, cor, extra=10 if b is vencedora else 0)
        x, y = int(b["center"]["x"]), int(b["box"]["y1"]) - 12
        cv2.putText(out, str(n), (x, y), cv2.FONT_HERSHEY_SIMPLEX, fonte, (0, 0, 0), int(esp * 1.8), cv2.LINE_AA)
        cv2.putText(out, str(n), (x, y), cv2.FONT_HERSHEY_SIMPLEX, fonte, cor, esp, cv2.LINE_AA)
    if balim:
        elipse(balim, (255, 255, 255))
        if vencedora:
            cv2.line(out, (int(balim["center"]["x"]), int(balim["center"]["y"])),
                     (int(vencedora["center"]["x"]), int(vencedora["center"]["y"])), (50, 230, 50), max(2, esp // 2), cv2.LINE_AA)
    return out

# --- reprodução do visual da API (cloud-run-function/main.py) ---
def _circulo_app(clean_img, img, inst, color, margin=10, stroke=None):
    if stroke is None:
        stroke = max(img.shape[0], img.shape[1]) * 0.005
    c, r = int(inst["center"]["x"]), int(inst["center"]["y"])
    c_radius = (inst["box"]["x2"] - inst["box"]["x1"]) / 2
    r_radius = (inst["box"]["y2"] - inst["box"]["y1"]) / 2
    rr, cc = skimage.draw.ellipse(r, c, int(r_radius + margin + stroke), int(c_radius + margin + stroke), shape=img.shape)
    img[rr, cc] = color
    rr, cc = skimage.draw.ellipse(r, c, int(r_radius + margin), int(c_radius + margin), shape=img.shape)
    img[rr, cc] = clean_img[rr, cc]

def imagens_do_app(img, bolas, vencedora, balim):
    img = img.copy()
    img_winner = img.copy()
    for i, ball in reversed(list(enumerate(bolas))):
        avg = np.array(img[ball["mask"]].mean(axis=0), dtype=np.uint8)
        img[ball["mask"]] = avg
        if i != 0:
            img_winner[ball["mask"]] = avg
    green = np.array([50, 230, 50], dtype=np.uint8)
    if vencedora:
        _circulo_app(img, img_winner, vencedora, green)
    if balim:
        stroke = max(img.shape[0], img.shape[1]) * 0.0022
        for start, end in (
            ((max(int(balim["box"]["y1"]), 0), max(int(balim["center"]["x"] - stroke / 2), 0)),
             (min(int(balim["box"]["y2"]), img.shape[0] - 1), min(int(balim["center"]["x"] + stroke / 2), img.shape[1] - 1))),
            ((max(int(balim["center"]["y"] - stroke / 2), 0), max(int(balim["box"]["x1"]), 0)),
             (min(int(balim["center"]["y"] + stroke / 2), img.shape[0] - 1), min(int(balim["box"]["x2"]), img.shape[1] - 1))),
        ):
            rr, cc = skimage.draw.rectangle(start=start, end=end)
            img[rr, cc] = green
            img_winner[rr, cc] = green
        _circulo_app(img, img_winner, balim, green, margin=0, stroke=stroke)
        _circulo_app(img.copy(), img, balim, green, margin=0, stroke=stroke)
    return img, img_winner

def gif_do_app(base, winner, caminho, lado_max=1280):
    fator = min(1.0, lado_max / max(base.shape[:2]))
    tam = (int(base.shape[1] * fator), int(base.shape[0] * fator))
    frames, p = [], 0.0
    for step in (0.25, -0.25):
        for _ in range(int(math.fabs(1.0 // step))):
            frames.append(Image.fromarray(np.array(base * (1 - p) + winner * p, dtype=np.uint8)).resize(tam))
            p += step
    frames[0].save(caminho, format="gif", append_images=frames[1:], save_all=True, duration=100, loop=0, optimize=True)
    return caminho

def julgar(caminho_foto, confianca=0.82, modelo=None):
    rec = modelos[modelo or next(iter(modelos))]
    img = Recognizer.read_file(caminho_foto)
    t = time.time()
    todas = [i for i in rec.predict_data(img) if i["score"] >= confianca]
    ms = (time.time() - t) * 1000
    bolas, vencedora, balim = processar_bolas(img, todas)
    return img, todas, bolas, vencedora, balim, ms

def legenda(todas, bolas, vencedora, balim):
    if balim is None:
        return f"sem decisão ({len(todas)} bola(s))"
    txt = f"{len(bolas)} bola(s) + balim | vencedora a {vencedora['distance']:.0f} px"
    if len(bolas) > 1:
        txt += f" | 2ª a {bolas[1]['distance']:.0f} px"
    return txt

## 4. Tela de teste
Rode a célula e use a tela que aparece logo abaixo. Escolha o **modelo** (A100 ou G4) e a **confiança** (começa em **0.82**, o valor de produção). A aba **Comparar modelos** mostra a mesma foto julgada pelos dois. Marque `COMPARTILHAR` para ganhar um link público temporário (72 h) e testar pelo celular na cancha. **Atenção:** qualquer pessoa com esse link consegue usar o modelo enquanto a sessão estiver aberta.

In [ ]:
import gradio as gr
import pandas as pd

COMPARTILHAR = False  # @param {type:"boolean"}

def analisar(caminho_foto, confianca, modelo):
    if caminho_foto is None:
        return None, None, None, "Envie uma foto da jogada.", None
    img, todas, bolas, vencedora, balim, ms = julgar(caminho_foto, confianca, modelo)
    analise = imagem_analise(img, todas, bolas, vencedora, balim)
    gif = None
    if balim:
        base, win = imagens_do_app(img, bolas, vencedora, balim)
        gif = gif_do_app(base, win, "/content/resultado.gif")
    tabela = pd.DataFrame([{
        "nº": n, "vencedora": "🏆" if b is vencedora else "",
        "distância (px)": round(b["distance"]), "área (px)": int(b["area"]), "score": round(b["score"], 2),
    } for n, b in enumerate(bolas, 1)])
    if balim is None:
        resumo = f"**{modelo}** ⚠️ Sem decisão: {len(todas)} bola(s) detectada(s) acima de {confianca:.2f}. O juiz precisa de pelo menos 2 bolas, com o balim fora da faixa de 10% das bordas. Tente baixar a confiança."
    else:
        descartadas = len(todas) - len(bolas) - 1
        resumo = (f"**{modelo}** ✅ {len(bolas)} bola(s) + balim | vencedora: nº 1, a {vencedora['distance']:.0f} px do balim"
                  + (f" | 2ª a {bolas[1]['distance']:.0f} px (diferença {bolas[1]['distance'] - vencedora['distance']:.0f} px)" if len(bolas) > 1 else "")
                  + (f" | {descartadas} detecção(ões) descartada(s) como sujeira/borda (cinza)" if descartadas > 0 else "")
                  + f" | predição em {ms:.0f} ms")
    # mesma foto em todos os modelos carregados
    comparacao = []
    for nome in modelos:
        r = julgar(caminho_foto, confianca, nome)
        comparacao.append((imagem_analise(*r[:5]), f"{nome}: {legenda(*r[1:5])} | {r[5]:.0f} ms"))
    return analise, gif, tabela, resumo, comparacao

with gr.Blocks(title="Juiz de Bocha — teste") as tela:
    gr.Markdown("### Juiz de Bocha — teste do modelo\nBranco = balim · Verde = vencedora · Amarelo = demais · Cinza = descartada")
    with gr.Row():
        with gr.Column(scale=1):
            foto = gr.Image(type="filepath", sources=["upload", "webcam", "clipboard"], label="Foto da jogada")
            modelo = gr.Radio(list(modelos), value=next(iter(modelos)), label="Modelo")
            conf = gr.Slider(0.3, 0.99, value=0.82, step=0.01, label="Confiança mínima")
            botao = gr.Button("Julgar", variant="primary")
        with gr.Column(scale=2):
            resumo = gr.Markdown()
            with gr.Tabs():
                with gr.Tab("Análise"):
                    saida = gr.Image(label="Análise", type="numpy")
                with gr.Tab("Animação do app"):
                    gif = gr.Image(label="GIF da API", type="filepath")
                with gr.Tab("Comparar modelos"):
                    comparacao = gr.Gallery(label="Mesma foto em cada modelo", columns=len(modelos), height="auto")
            tabela = gr.Dataframe(label="Bolas (ordem de distância ao balim)")
    entradas, saidas = [foto, conf, modelo], [saida, gif, tabela, resumo, comparacao]
    botao.click(analisar, entradas, saidas)
    foto.change(analisar, entradas, saidas)
    modelo.change(analisar, entradas, saidas)
    conf.release(analisar, entradas, saidas)

tela.launch(share=COMPARTILHAR, debug=False)

## 5. Galeria — várias fotos de uma vez
Roda os modelos em até `QUANTIDADE` fotos de uma pasta do Drive: **uma linha por foto, uma coluna por modelo**. Por padrão usa a pasta `bocha` usada no treino; troque por uma pasta com **fotos novas** (que o modelo nunca viu) para um teste honesto.

In [ ]:
import matplotlib.pyplot as plt

PASTA_FOTOS = "/content/drive/MyDrive/BACKUP PENDRIVE 256/CursoSegImgPython/bocha"  # @param {type:"string"}
QUANTIDADE = 16  # @param {type:"integer"}
CONFIANCA = 0.82  # @param {type:"number"}

fotos = sorted(f for f in os.listdir(PASTA_FOTOS)
               if not f.startswith("._") and f.lower().endswith((".jpg", ".jpeg", ".png")))[:QUANTIDADE]
nomes = list(modelos)
fig, axs = plt.subplots(len(fotos), len(nomes), figsize=(9 * len(nomes), 6.5 * len(fotos)), squeeze=False)
decididas = {n: 0 for n in nomes}
tempos = {n: [] for n in nomes}
for linha, foto_nome in enumerate(fotos):
    for col, nome in enumerate(nomes):
        img, todas, bolas, vencedora, balim, ms = julgar(f"{PASTA_FOTOS}/{foto_nome}", CONFIANCA, nome)
        decididas[nome] += balim is not None
        tempos[nome].append(ms)
        ax = axs[linha][col]
        ax.imshow(imagem_analise(img, todas, bolas, vencedora, balim)); ax.axis("off")
        ax.set_title(f"{nome} — {foto_nome[:24]}\n{legenda(todas, bolas, vencedora, balim)}", fontsize=11)
plt.tight_layout(); plt.show()
for nome in nomes:
    print(f"{nome}: decidiu em {decididas[nome]}/{len(fotos)} fotos | predição média {np.mean(tempos[nome][1:] or tempos[nome]):.0f} ms (confiança {CONFIANCA})")